# 03 — Data Integration

Load cleaned datasets and:
- Discover value-based join connections across all 14 tables
- Confirm pairwise linkages (ACH- IDs, PR- IDs, ENSG IDs, CVCL accessions)
- Save integrated / processed outputs to `data/processed/`

In [5]:
import sys, os
#sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', 'scripts'))
sys.path.insert(0, "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/src/scripts")

import re
import pandas as pd
from itertools import combinations
from data_utils import load_clean_parquets
from export_data import export_processed_csvs, export_clean_report

## Load Cleaned Data

In [6]:
tables = load_clean_parquets()

Loaded hpa_rna: (24315372, 6)
Loaded depmap_expr: (1495, 53961)
Loaded geo_expr: (19914, 3268)
Loaded proteomics: (375, 12559)
Loaded protein_map: (12558, 3)
Loaded fusions: (184237, 32)
Loaded mutations: (1066869, 70)
Loaded cellosaurus: (152231, 17)
Loaded depmap_profiles: (3830, 5)
Loaded sample_info: (1840, 29)
Loaded geo_info: (3267, 23)
Loaded hpa_desc: (1206, 7)
Loaded metabolomics: (928, 227)
Loaded mirna: (734, 956)
Loaded signatures: (3021, 12)


## Value-Based Join Discovery

Find connections between tables by comparing actual values in candidate key columns.

In [7]:
ID_HINTS = re.compile(
    r"(id|name|accession|gsm|cvcl|ach|pr-|profile|model|gene|cell|symbol|ensembl|ccle|rrid|cosmic)",
    re.IGNORECASE,
)

def get_key_columns(df: pd.DataFrame, sample_n: int = 200_000) -> list:
    if len(df) > sample_n:
        df = df.sample(sample_n, random_state=0)
    return [
        col for col in df.columns
        if (bool(ID_HINTS.search(str(col))) or df[col].dtype == object)
        and df[col].nunique(dropna=True) >= 2
    ]


def build_fingerprints(tables: dict, sample_n: int = 200_000) -> dict:
    fingerprints = {}
    for name, df in tables.items():
        key_cols = get_key_columns(df, sample_n=sample_n)
        df_use = df.sample(sample_n, random_state=0) if len(df) > sample_n else df
        col_sets = {
            col: set(df_use[col].dropna().astype(str).str.strip().str.lower().unique())
            for col in key_cols
        }
        fingerprints[name] = col_sets
        print(f"{name:25s}: {len(col_sets)} candidate key columns")
    return fingerprints


def find_connections(fingerprints: dict, min_overlap: int = 5, min_pct: float = 5.0) -> pd.DataFrame:
    rows = []
    for t1, t2 in combinations(fingerprints.keys(), 2):
        for col1, set1 in fingerprints[t1].items():
            for col2, set2 in fingerprints[t2].items():
                if not set1 or not set2:
                    continue
                inter = set1 & set2
                n = len(inter)
                if n < min_overlap:
                    continue
                pct = n / min(len(set1), len(set2)) * 100
                if pct < min_pct:
                    continue
                rows.append({
                    "table_1": t1, "column_1": col1,
                    "table_2": t2, "column_2": col2,
                    "n_overlap": n,
                    "n_distinct_1": len(set1), "n_distinct_2": len(set2),
                    "pct_of_smaller": round(pct, 1),
                    "example_shared": list(inter)[:3],
                })
    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.sort_values(["pct_of_smaller", "n_overlap"], ascending=False).reset_index(drop=True)
    return result

In [8]:
print("Building value fingerprints...")
fingerprints = build_fingerprints(tables)

Building value fingerprints...
hpa_rna                  : 3 candidate key columns
depmap_expr              : 0 candidate key columns
geo_expr                 : 3268 candidate key columns
proteomics               : 7 candidate key columns
protein_map              : 2 candidate key columns
fusions                  : 12 candidate key columns
mutations                : 18 candidate key columns
cellosaurus              : 7 candidate key columns
depmap_profiles          : 3 candidate key columns
sample_info              : 15 candidate key columns
geo_info                 : 9 candidate key columns
hpa_desc                 : 2 candidate key columns
metabolomics             : 18 candidate key columns
mirna                    : 241 candidate key columns
signatures               : 6 candidate key columns


In [9]:
print("Finding value-based connections...")
connections = find_connections(fingerprints, min_overlap=5, min_pct=5.0)
print(f"Found {len(connections)} candidate join connections.")
connections

Finding value-based connections...
Found 88 candidate join connections.


,table_1,column_1,table_2,column_2,n_overlap,n_distinct_1,n_distinct_2,pct_of_smaller,example_shared
0,hpa_rna,cell line,hpa_desc,cell line,1206,1206,1206,100.0,"[t3m-10, panc-1, ovcar-5]"
1,sample_info,depmap_id,metabolomics,depmap_id,927,1840,927,100.0,"[ach-000213, ach-000905, ach-000148]"
2,cellosaurus,cellosaurus_accession,geo_info,cellosaurus_id,797,152231,797,100.0,"[cvcl_0839, cvcl_6022, cvcl_m572]"
3,proteomics,depmap_id,sample_info,depmap_id,375,375,1840,100.0,"[ach-000213, ach-000374, ach-000008]"
4,cellosaurus,cellosaurus_accession,sample_info,rrid,1812,152231,1814,99.9,"[cvcl_2412, cvcl_1421, cvcl_1661]"
...,...,...,...,...,...,...,...,...,...
83,mirna,katoiii_stomach,signatures,aneuploidy,5,261,40,12.5,"[10.0, 5.0, 20.0]"
84,cellosaurus,cellosaurus_cell_line_name,geo_info,source_name_ch1,57,152160,537,10.6,"[mkn7, mkn74, sn12c]"
85,sample_info,cell_line_name,geo_info,source_name_ch1,52,1747,537,9.7,"[mkn7, mcf12a, mkn74]"
86,hpa_rna,cell line,geo_info,source_name_ch1,48,1206,537,8.9,"[mkn7, mkn74, sk-mel-28]"


## Confirm Pairwise Key Linkages

In [10]:
sample_info    = tables["sample_info"]
depmap_profiles= tables["depmap_profiles"]
mutations      = tables["mutations"]
depmap_expr    = tables["depmap_expr"]

# ACH- linkage: sample_info <-> depmap_profiles
achs_si   = set(sample_info["depmap_id"].dropna())
achs_prof = set(depmap_profiles["modelid"].dropna())
print(f"sample_info ACH IDs:       {len(achs_si):>5,}")
print(f"depmap_profiles ACH IDs:   {len(achs_prof):>5,}")
print(f"Overlap:                   {len(achs_si & achs_prof):>5,}")
print()

# PR- linkage: depmap_profiles <-> mutations
pr_prof = set(depmap_profiles["profileid"].dropna())
pr_mut  = set(mutations["profileid"].dropna()) if "profileid" in mutations.columns else set()
print(f"depmap_profiles PR IDs:    {len(pr_prof):>5,}")
print(f"mutations PR IDs:          {len(pr_mut):>5,}")
print(f"Overlap:                   {len(pr_prof & pr_mut):>5,}")

sample_info ACH IDs:       1,840
depmap_profiles ACH IDs:   1,822
Overlap:                   1,749

depmap_profiles PR IDs:    3,830
mutations PR IDs:          2,828
Overlap:                   2,329


## Save Column-Level Data Quality Report

In [11]:
export_clean_report(tables)

Written 15 sheets to: /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/outputs/clean_data_report.xlsx


## Export Processed Files to data/processed/

In [12]:
# Export all cleaned tables as CSV to the processed directory
export_processed_csvs(tables)

Exported hpa_rna -> /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/outputs/processed/hpa_rna.csv
Exported depmap_expr -> /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/outputs/processed/depmap_expr.csv
Exported geo_expr -> /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/outputs/processed/geo_expr.csv
Exported proteomics -> /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/outputs/processed/proteomics.csv
Exported protein_map -> /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/outputs/processed/protein_map.csv
Exported fusions -> /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/outputs/processed/fusions.csv
Exported mutations -> /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/outputs/processed/mutations.csv
Exported cellosaurus -> /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/outputs/proces